In [87]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

import pandas as pd
import numpy as np

import tensorflow as tf

from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

from staticvectors import StaticVectors
from tensorflow.keras.initializers import Constant
from tensorflow.keras import models, layers
from tensorflow.data import Dataset

import re

In [88]:
file_path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
    "IMDB Dataset.csv",
    pandas_kwargs={"encoding": "latin-1"}
)

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [89]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

X = df['review'].values
y = df['sentiment'].values.astype(np.int32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [90]:
def clean_text(input_data):
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, r'<[^>]+>', ' ')
    return tf.strings.regex_replace(stripped_html, r'[^a-z\s]', '')

X_train_clean = [clean_text(t) for t in X_train]
X_test_clean = [clean_text(t) for t in X_test]

vector_layer = TextVectorization(
    standardize=clean_text,
    max_tokens=15000,
    ngrams=1,
    output_sequence_length=200,
    output_mode='int'
)

vector_layer.adapt(X_train_clean)

In [91]:
gloveModel = StaticVectors("neuml/glove-6B")
# glove.embeddings(["word"])

In [92]:
vocabulary = vector_layer.get_vocabulary()
embedding_dim = gloveModel.embeddings(['the']).shape[1]
embedding_matrix = np.zeros((len(vocabulary), embedding_dim))

for i, word in enumerate(vocabulary):
    embedding_matrix[i] = gloveModel.embeddings([word])[0]

In [93]:
embedding_layer = Embedding(
    input_dim=len(vocabulary),
    output_dim=embedding_dim,
    embeddings_initializer=Constant(embedding_matrix),
    trainable=True,
    mask_zero=True
)

model = models.Sequential([
    layers.Input(shape=(), dtype=tf.string),
    vector_layer,
    embedding_layer,
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    # Dense(1, activation='relu')
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_10           │ (None, 200)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_8 (Embedding)         │ (None, 200, 300)       │     4,500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_8      │ (None, 300)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │        19,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,519,329 (17.24 MB)

 Trainable params: 4,519,329 (17.24 MB)

 Non-trainable params: 0 (0.00 B)

In [94]:
history = model.fit(
    tf.constant(X_train),
    y_train,
    validation_data=(tf.constant(X_test), y_test),
    epochs=4,
    batch_size=128
)

Epoch 1/4
313/313 ━━━━━━━━━━━━━━━━━━━━ 54s 164ms/step - accuracy: 0.8106 - loss: 0.4212 - val_accuracy: 0.8746 - val_loss: 0.2944
Epoch 2/4
313/313 ━━━━━━━━━━━━━━━━━━━━ 45s 144ms/step - accuracy: 0.8983 - loss: 0.2539 - val_accuracy: 0.8826 - val_loss: 0.2804
Epoch 3/4
313/313 ━━━━━━━━━━━━━━━━━━━━ 45s 143ms/step - accuracy: 0.9215 - loss: 0.2069 - val_accuracy: 0.8726 - val_loss: 0.3004
Epoch 4/4
313/313 ━━━━━━━━━━━━━━━━━━━━ 47s 148ms/step - accuracy: 0.9356 - loss: 0.1782 - val_accuracy: 0.8750 - val_loss: 0.3055


In [95]:
reviews = np.array([
    'Really interesting i liked it!!!!',
    'Not interesting at all I will not recommend it'
])

predictions = model.predict(tf.constant(reviews))

for review, pred in zip(reviews, predictions):
    prob = pred[0]
    sentiment = 'Positive' if pred[0] > 0.5 else 'Negative'
    confidence = prob if sentiment == 'Positive' else 1 - prob
    print(f'Review - {review},\n'
          f'Sentiment - {sentiment},\n'
          f'Score - {prob},\n'
          f'Confidence - {confidence}\n'
          f'-----------------------------\n')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 324ms/step
Review - Really interesting i liked it!!!!,
Sentiment - Positive,
Score - 0.9705263376235962,
Confidence - 0.9705263376235962
-----------------------------

Review - Not interesting at all I will not recommend it,
Sentiment - Negative,
Score - 0.4386255741119385,
Confidence - 0.5613744258880615
-----------------------------

